In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
dir_path = Path(r"C:\Users\yu2hy\OneDrive\◎2020_copus\가공_결과물\바른형태분석_말뭉치")
sen_folder = dir_path / "최신버전" / "sen"

for file in v_folder.glob('*.csv'):
    print(file)

#%% 1.2 문장 파일 읽어오기
print('%%%%%1.2 문장 파일-sen2 읽기') 
df_sen_list = []
for file in sen_folder.glob("*.csv"):
    with open(file, "r", encoding = "UTF-8") as f:
        df = pd.read_csv(f, low_memory=False)
        #'Unnamed:'를 포함하는 columns 삭제
        cols_to_drop = [col for col in df.columns if 'Unnamed:' in col]
        df = df.drop(columns=cols_to_drop)
        df_sen_list.append(df)
df_sen_list[2].rename(columns={'docu_id': 'file_id'}, inplace=True)

###
# 문장 단위 DataFrame이 있다고 가정
# df_sen에는 적어도 ['file_id', 'sen_id', 'form'] 열이 있어야 함

# 1. 전체 문장 수
total_sentences = len(df_sen_list[2])

# 2. file_id별 비율 계산
file_counts = df_sen_list[2]['file_id'].value_counts()
file_ratios = file_counts / total_sentences

# 3. 각 file_id에서 샘플링할 문장 수 계산 (비율 기반)
sample_counts = (file_ratios * 250).round().astype(int)

# 총합이 250이 안 될 수 있으므로, 부족한 수를 보완
diff = 250 - sample_counts.sum()
if diff > 0:
    # 문장 수가 많은 file_id 중심으로 보완
    top_file_ids = file_counts.sort_values(ascending=False).index
    for file_id in top_file_ids:
        sample_counts[file_id] += 1
        diff -= 1
        if diff == 0:
            break

# 4. 각 file_id에서 해당 수만큼 랜덤 샘플링
sampled_list = []
for file_id, count in sample_counts.items():
    subset = df_sen_list[2][df_sen_list[2]['file_id'] == file_id]
    if len(subset) < count:
        sampled = subset  # 부족할 경우 그냥 전부 사용
    else:
        sampled = subset.sample(n=count, random_state=42)
    sampled_list.append(sampled)

# 5. 하나로 합치기
sampled_df = pd.concat(sampled_list).sort_values(by=['file_id', 'sen_id'])

# 결과 확인
print(sampled_df[['file_id', 'sen_id', 'form']].head())
sampled_df.to_csv('sample_sejong.csv')